In [1]:
import duckdb

# Your real store root (same path run_daily_stats.py uses).
PARSED_ROOT = ("/Users/shazzak/Library/CloudStorage/"
               "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")

# All trades partitions: <root>/trades/date=YYYY-MM-DD/*.parquet
trades_glob = f"{PARSED_ROOT}/trades/*/*.parquet"

query = f"""
SELECT
  -- strip the -JUL / -JULB suffix to get the underlying (OGDC-JUL -> OGDC)
  regexp_replace(symbol, '-.*$', '') AS underlying,
  segment,
  count(DISTINCT date)          AS days_traded,
  count(*)                      AS trades,
  round(sum(qty * price) / 1e6, 1) AS notional_m
FROM read_parquet('{trades_glob}', hive_partitioning=true)
WHERE segment IN ('STOCK_DEL_FUT', 'STOCK_CS_FUT')
GROUP BY 1, 2
ORDER BY underlying, notional_m DESC
"""

df = duckdb.sql(query).df()

# Show the whole result in the notebook.
import pandas as pd
pd.set_option("display.max_rows", None, "display.width", 200)
print(f"{len(df)} rows")
df

0 rows


,underlying,segment,days_traded,trades,notional_m


In [ ]:
df.tail()


In [2]:
import duckdb, glob
staged = len(glob.glob("daily_stats_staging/daily_stats_*.parquet"))
n, d, lo, hi = duckdb.sql("""
    SELECT count(*), count(DISTINCT date), min(date), max(date)
    FROM 'daily_stats_ALL.parquet'
""").fetchone()
print(f"staging files: {staged}")
print(f"merged: {n:,} rows | {d} distinct dates | {lo} -> {hi}")
print("MATCH" if staged == d else "MISMATCH -- do not delete")

staging files: 207
merged: 122,912 rows | 207 distinct dates | 2025-09-01 -> 2026-06-30
MATCH


In [3]:
import duckdb
duckdb.sql("""
    SELECT date, count(*) AS symbols, round(sum(notional_m),1) AS notional_m
    FROM 'daily_stats_ALL.parquet'
    GROUP BY date ORDER BY symbols
    LIMIT 10
""").df()

,date,symbols,notional_m
0,2026-03-09,538,44728.0
1,2026-03-06,539,29271.9
2,2026-03-19,542,15184.7
3,2026-04-03,545,19821.4
4,2026-03-12,548,31416.5
5,2026-03-13,549,22872.9
6,2025-12-03,550,57194.1
7,2025-11-07,550,38785.1
8,2026-03-31,552,29663.0
9,2025-12-01,553,43292.4


In [4]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")

duckdb.sql(f"""
    SELECT date,
           min(transact_time) AS first_trade,
           max(transact_time) AS last_trade,
           round(date_diff('minute', min(transact_time), max(transact_time)) / 60.0, 2) AS session_hours,
           count(*) AS trades
    FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
    WHERE initiator <> 'AUCTION'
    GROUP BY date
    ORDER BY session_hours
    LIMIT 20
""").df()

,date,first_trade,last_trade,session_hours,trades
0,2026-03-13,2026-03-13 09:17:00.010000+05:00,2026-03-13 12:29:59.940000+05:00,3.20,210486
1,2026-02-20,2026-02-20 09:17:00.100000+05:00,2026-02-20 12:29:59.900000+05:00,3.20,329010
2,2026-02-27,2026-02-27 09:17:00.010000+05:00,2026-02-27 12:29:59.910000+05:00,3.20,386717
3,2026-03-06,2026-03-06 09:17:00.030000+05:00,2026-03-06 12:29:59.920000+05:00,3.20,296241
4,2026-03-19,2026-03-19 09:25:49.160000+05:00,2026-03-19 13:29:59.970000+05:00,4.07,141560
5,2026-03-12,2026-03-12 09:17:00.040000+05:00,2026-03-12 13:29:59.990000+05:00,4.20,267925
6,2026-02-19,2026-02-19 09:17:00.010000+05:00,2026-02-19 13:29:59.980000+05:00,4.20,341845
7,2026-02-24,2026-02-24 09:17:00.010000+05:00,2026-02-24 13:29:59.980000+05:00,4.20,484402
8,2026-03-02,2026-03-02 09:17:00.010000+05:00,2026-03-02 13:29:59.920000+05:00,4.20,370596
9,2026-03-11,2026-03-11 09:17:00.070000+05:00,2026-03-11 13:29:59.880000+05:00,4.20,308129


In [5]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")

# Build a session calendar: elapsed span, actual traded minutes, largest intra-day
# gap (a break shows up as a large gap), and day of week.
cal = duckdb.sql(f"""
    WITH t AS (
        SELECT date, transact_time AS ts
        FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
        WHERE initiator <> 'AUCTION'
    ),
    g AS (
        SELECT date, ts,
               date_diff('second', lag(ts) OVER (PARTITION BY date ORDER BY ts), ts) AS gap_s
        FROM t
    )
    SELECT date,
           dayname(date)                                        AS dow,
           min(ts)                                              AS open_ts,
           max(ts)                                              AS close_ts,
           round(date_diff('second', min(ts), max(ts))/3600.0, 2) AS elapsed_h,
           round(max(gap_s)/60.0, 1)                            AS max_gap_min,
           round((date_diff('second', min(ts), max(ts)) - coalesce(max(gap_s),0))/3600.0, 2) AS traded_h,
           count(*)                                             AS trades
    FROM g
    GROUP BY date
    ORDER BY date
""").df()

# Classify the regime from the measured session, not from a calendar assumption.
cal["regime"] = "normal"
cal.loc[(cal.dow == "Friday"), "regime"] = "friday"
ram = (cal.date >= "2026-02-19") & (cal.date <= "2026-03-19")
cal.loc[ram & (cal.dow != "Friday"), "regime"] = "ramadan"
cal.loc[ram & (cal.dow == "Friday"), "regime"] = "ramadan_friday"

print(cal.groupby("regime")[["elapsed_h", "traded_h", "max_gap_min", "trades"]].median())
cal.to_parquet("session_calendar.parquet", index=False)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                elapsed_h  traded_h  max_gap_min    trades
regime                                                    
friday               7.22      4.68        152.0  483524.5
normal               5.97      5.96          0.5  495123.5
ramadan              4.22      4.21          0.5  341845.0
ramadan_friday       3.22      3.21          0.5  312625.5


In [6]:
import duckdb, pandas as pd
pd.set_option("display.width", 220, "display.max_columns", None)

PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")
D   = "2025-09-23"
SYM = "DMC"

SNAP = f"{PARSED}/ob_snapshot/date={D}/*.parquet"
TRD  = f"{PARSED}/trades/date={D}/*.parquet"

# 1. THE DIRECT CONFIRMATION: which entry_types exist, at which levels?
#    If OFFER never appears at level 1, the pivot has no OFFER column -> KeyError.
print("=== 1. entry_type x level census ===")
print(duckdb.sql(f"""
    SELECT entry_type, level, count(*) AS rows,
           count(DISTINCT msg_seq) AS msgs,
           min(px) AS min_px, max(px) AS max_px, sum(qty) AS tot_qty
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
    GROUP BY entry_type, level
    ORDER BY entry_type, level
""").df().to_string(index=False))

# 2. Level 1 only, both sides side by side. A zero OFFER count is the smoking gun.
print("\n=== 2. level-1 BID vs OFFER ===")
print(duckdb.sql(f"""
    SELECT
      count(*) FILTER (WHERE entry_type='BID'   AND level=1) AS bid_l1_rows,
      count(*) FILTER (WHERE entry_type='OFFER' AND level=1) AS offer_l1_rows,
      count(DISTINCT msg_seq) AS total_snapshot_msgs
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
""").df().to_string(index=False))

# 3. Session shape: phase / trading_status over the day, and whether it was suspended.
print("\n=== 3. phase & status ===")
print(duckdb.sql(f"""
    SELECT phase, trading_status, suspended_all_day, break_reason,
           count(DISTINCT msg_seq) AS msgs,
           min(orig_time) AS first_msg, max(orig_time) AS last_msg
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
    GROUP BY phase, trading_status, suspended_all_day, break_reason
    ORDER BY msgs DESC
""").df().to_string(index=False))

# 4. WAS IT LOCKED LIMIT-UP? Compare the best bid against the upper circuit limit.
#    A bid sitting at/above the cap with no offers is the classic one-sided cause.
print("\n=== 4. circuit limits vs best bid ===")
print(duckdb.sql(f"""
    SELECT
      max(px) FILTER (WHERE entry_type='UPPER_CIRCUIT_BREAKER') AS limit_up,
      max(px) FILTER (WHERE entry_type='LOWER_CIRCUIT_BREAKER') AS limit_dn,
      max(px) FILTER (WHERE entry_type='BID'   AND level=1)     AS best_bid,
      max(px) FILTER (WHERE entry_type='OFFER' AND level=1)     AS best_ask,
      max(prev_close)                                           AS prev_close
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
""").df().to_string(index=False))

# 5. The 201 shares: were they auction prints (no continuous aggressor) or continuous?
print("\n=== 5. the trades ===")
print(duckdb.sql(f"""
    SELECT initiator, aggressor_side, count(*) AS n, sum(qty) AS shares,
           min(price) AS min_px, max(price) AS max_px,
           min(transact_time) AS first, max(transact_time) AS last
    FROM read_parquet('{TRD}')
    WHERE symbol = '{SYM}'
    GROUP BY initiator, aggressor_side
    ORDER BY n DESC
""").df().to_string(index=False))

# 6. CONTROL: a healthy symbol the same day, to prove the query itself is sound.
print("\n=== 6. control (OGDC, same day) ===")
print(duckdb.sql(f"""
    SELECT
      count(*) FILTER (WHERE entry_type='BID'   AND level=1) AS bid_l1_rows,
      count(*) FILTER (WHERE entry_type='OFFER' AND level=1) AS offer_l1_rows
    FROM read_parquet('{SNAP}')
    WHERE symbol = 'OGDC'
""").df().to_string(index=False))

=== 1. entry_type x level census ===
           entry_type  level  rows  msgs  min_px  max_px  tot_qty
              AGG_BID      0    19    19   80.67   80.67  20879.0
                  BID      1    19    19   80.67   80.67  20879.0
           LAST_TRADE      0    11    11   80.67   80.67   1205.0
LOWER_CIRCUIT_BREAKER      0    28    28   66.01   66.01      0.0
         NET_CHANGE_1      0    11    11    7.33    7.33      0.0
         NET_CHANGE_2      0    11    11    0.00    7.33      0.0
        OPENING_PRICE      0    11    11   80.67   80.67      0.0
         SESSION_HIGH      0    11    11   80.67   80.67      0.0
          SESSION_LOW      0    11    11   80.67   80.67      0.0
UPPER_CIRCUIT_BREAKER      0    28    28   80.67   80.67      0.0

=== 2. level-1 BID vs OFFER ===
 bid_l1_rows  offer_l1_rows  total_snapshot_msgs
          19              0                   28

=== 3. phase & status ===
             phase trading_status  suspended_all_day   break_reason  msgs      